# 2 — Train, Evaluate & Visualize

Complete experiment in one notebook:

1. **Data** — Load VQA v2.0 dataset
2. **Models** — Define encoders, attention blocks, and full VQA models
3. **Training** — Train symmetric baseline and asymmetric model
4. **Evaluation** — Compare metrics (Top-1, Top-5 accuracy)
5. **Visualization** — Training curves, attention heatmaps, qualitative examples

**Run all cells in order.** Edit the configuration cell below to change hyperparameters.

In [43]:
!pip install -q torch torchvision transformers matplotlib tqdm Pillow

In [44]:
import json
import random
import time
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ViT_B_16_Weights, vit_b_16
from tqdm import tqdm
from transformers import RobertaModel, RobertaTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


## Configuration

Edit these variables to change the experiment. Set `MAX_SAMPLES = 1000` for a quick dev run.

In [45]:
# Paths
DATA_DIR       = Path("../data")
CHECKPOINT_DIR = Path("../results/checkpoints")
METRICS_DIR    = Path("../results/metrics")
FIGURES_DIR    = Path("../results/figures")

# Model
NUM_ANSWERS      = 1000
EMBED_DIM        = 512
NUM_HEADS        = 8
DROPOUT          = 0.3
FREEZE_ENCODERS  = True

# Data
MAX_QUESTION_LEN = 20
MAX_SAMPLES      = None    # set to 1000 for a quick dev run

# Training
BATCH_SIZE       = 64
LEARNING_RATE    = 1e-4
WEIGHT_DECAY     = 1e-5
EPOCHS           = 20
NUM_WORKERS      = 4
SEED             = 42
USE_AMP          = True

for d in [CHECKPOINT_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

---
## 1. Data Loading

In [46]:
def build_answer_vocab(annotations_file, top_k=1000):
    """Build answer vocabulary from the top_k most frequent answers."""
    with open(annotations_file) as f:
        annotations = json.load(f)["annotations"]

    counter = Counter()
    for ann in annotations:
        counter[ann["multiple_choice_answer"]] += 1

    most_common = [ans for ans, _ in counter.most_common(top_k)]
    answer_to_idx = {ans: idx for idx, ans in enumerate(most_common)}
    idx_to_answer = {idx: ans for ans, idx in answer_to_idx.items()}
    return answer_to_idx, idx_to_answer


def get_image_transform(split="train"):
    """ImageNet-normalised transform for ViT-B/16 (224x224)."""
    if split == "train":
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

In [47]:
class VQADataset(Dataset):
    """PyTorch dataset for VQA v2.0.

    Returns (image_tensor, input_ids, attention_mask, answer_idx) tuples.
    Only keeps samples whose answer is in the top-K vocabulary.
    """

    def __init__(self, questions_file, annotations_file, image_dir,
                 answer_to_idx=None, top_k_answers=1000,
                 max_question_len=20, transform=None, max_samples=None):
        self.image_dir = Path(image_dir)
        self.max_question_len = max_question_len
        self.transform = transform or get_image_transform("val")
        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

        if answer_to_idx is None:
            self.answer_to_idx, self.idx_to_answer = build_answer_vocab(
                annotations_file, top_k_answers)
        else:
            self.answer_to_idx = answer_to_idx
            self.idx_to_answer = {v: k for k, v in answer_to_idx.items()}

        with open(questions_file) as f:
            questions_data = json.load(f)["questions"]
        with open(annotations_file) as f:
            annotations_data = json.load(f)["annotations"]

        ann_by_qid = {ann["question_id"]: ann for ann in annotations_data}

        self.samples = []
        for q in questions_data:
            ann = ann_by_qid.get(q["question_id"])
            if ann is None:
                continue
            answer = ann["multiple_choice_answer"]
            if answer not in self.answer_to_idx:
                continue
            self.samples.append({
                "question": q["question"],
                "image_id": q["image_id"],
                "answer_idx": self.answer_to_idx[answer],
            })
            if max_samples is not None and len(self.samples) >= max_samples:
                break

    def __len__(self):
        return len(self.samples)

    def _image_path(self, image_id):
        split = self.image_dir.name  # e.g. "train2014"
        return self.image_dir / f"COCO_{split}_{image_id:012d}.jpg"

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = Image.open(self._image_path(sample["image_id"])).convert("RGB")
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        answer_idx = torch.tensor(sample["answer_idx"], dtype=torch.long)
        return image, input_ids, attention_mask, answer_idx

In [48]:
# Build answer vocab from training annotations
train_ann = DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json"
answer_to_idx, idx_to_answer = build_answer_vocab(train_ann, NUM_ANSWERS)
print(f"Answer vocab: {len(answer_to_idx)} classes")

# Create datasets
train_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json",
    annotations_file=train_ann,
    image_dir=DATA_DIR / "images" / "train2014",
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("train"),
    max_samples=MAX_SAMPLES,
)
val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    image_dir=DATA_DIR / "images" / "val2014",
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("val"),
    max_samples=MAX_SAMPLES,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

# Save answer vocab
with open(CHECKPOINT_DIR / "answer_vocab.json", "w") as f:
    json.dump(answer_to_idx, f)

Answer vocab: 1000 classes
Train: 388,158 samples (6065 batches)
Val:   186,489 samples (2914 batches)


---
## 2. Model Definitions

### Encoders

Both encoders are **frozen** (pretrained weights fixed). Only the fusion layers and classifier train.

- **ImageEncoder** — ViT-B/16: produces 197 patch embeddings (196 spatial + 1 CLS), projected from 768 → `EMBED_DIM`
- **TextEncoder** — RoBERTa-base: produces per-token embeddings, projected from 768 → `EMBED_DIM`

In [49]:
class ImageEncoder(nn.Module):
    """ViT-B/16 image encoder. Output: (B, 197, embed_dim)."""

    VIT_HIDDEN_DIM = 768

    def __init__(self, embed_dim=512, freeze=True):
        super().__init__()
        vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        self.conv_proj = vit.conv_proj
        self.class_token = vit.class_token
        self.encoder = vit.encoder
        self.projection = nn.Linear(self.VIT_HIDDEN_DIM, embed_dim)

        if freeze:
            for param in self.conv_proj.parameters():
                param.requires_grad = False
            self.class_token.requires_grad = False
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, images):
        x = self.conv_proj(images)                         # (B, 768, 14, 14)
        x = x.flatten(2).transpose(1, 2)                   # (B, 196, 768)
        cls = self.class_token.expand(x.shape[0], -1, -1)  # (B, 1, 768)
        x = torch.cat([cls, x], dim=1)                     # (B, 197, 768)
        x = self.encoder(x)                                # (B, 197, 768)
        return self.projection(x)                          # (B, 197, embed_dim)


class TextEncoder(nn.Module):
    """RoBERTa-base text encoder. Output: (B, seq_len, embed_dim)."""

    ROBERTA_HIDDEN_DIM = 768

    def __init__(self, embed_dim=512, freeze=True):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.projection = nn.Linear(self.ROBERTA_HIDDEN_DIM, embed_dim)

        if freeze:
            for param in self.roberta.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(outputs.last_hidden_state)  # (B, seq_len, embed_dim)

### Cross-Attention Blocks

The core research contribution. Cross-attention lets one modality "ask questions" of the other:

- **Asymmetric**: Two **independent** blocks with separate weights — one for image→text, one for text→image
- **Symmetric** (baseline): A **single shared** block used in both directions — cannot learn directional patterns

In [50]:
class CrossAttentionBlock(nn.Module):
    """Cross-attention: queries from one modality attend to keys/values from another.
    Includes LayerNorm, residual connections, and a feed-forward network."""

    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, key_value, key_padding_mask=None):
        # Pre-norm cross-attention + residual
        q = self.norm_q(query)
        kv = self.norm_kv(key_value)
        attended, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=key_padding_mask,
            need_weights=True, average_attn_weights=True)
        query = query + attended
        # Pre-norm feed-forward + residual
        query = query + self.ff(self.norm_ff(query))
        return query, attn_weights


class AsymmetricCrossModalFusion(nn.Module):
    """Two SEPARATE cross-attention blocks with independent weights.
    Block 1: image attends to text (a <- b)
    Block 2: text attends to image (b <- a)"""

    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn_img_to_txt = CrossAttentionBlock(embed_dim, num_heads, dropout)
        self.cross_attn_txt_to_img = CrossAttentionBlock(embed_dim, num_heads, dropout)

    def forward(self, image_features, text_features, text_padding_mask=None):
        img_attended, attn_i2t = self.cross_attn_img_to_txt(
            query=image_features, key_value=text_features,
            key_padding_mask=text_padding_mask)
        txt_attended, attn_t2i = self.cross_attn_txt_to_img(
            query=text_features, key_value=image_features)
        return img_attended, txt_attended, attn_i2t, attn_t2i


class SymmetricCrossModalFusion(nn.Module):
    """ONE SHARED cross-attention block used in both directions (baseline)."""

    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.shared_cross_attn = CrossAttentionBlock(embed_dim, num_heads, dropout)

    def forward(self, image_features, text_features, text_padding_mask=None):
        img_attended, attn_i2t = self.shared_cross_attn(
            query=image_features, key_value=text_features,
            key_padding_mask=text_padding_mask)
        txt_attended, attn_t2i = self.shared_cross_attn(
            query=text_features, key_value=image_features)
        return img_attended, txt_attended, attn_i2t, attn_t2i

### Full VQA Models

Pipeline: encode image + text → cross-modal fusion → mean-pool → concatenate → MLP classifier

The two models are identical except for the fusion layer.

In [51]:
class AsymmetricVQAModel(nn.Module):
    """VQA model with asymmetric (two independent blocks) cross-attention."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3, freeze_encoders=True):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim, freeze=freeze_encoders)
        self.text_encoder = TextEncoder(embed_dim, freeze=freeze_encoders)
        self.fusion = AsymmetricCrossModalFusion(embed_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(embed_dim, num_answers))

    def forward(self, images, input_ids, attention_mask):
        img = self.image_encoder(images)
        txt = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0  # True = padding

        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(img, txt, text_pad_mask)

        z = torch.cat([img_att.mean(dim=1), txt_att.mean(dim=1)], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


class SymmetricVQAModel(nn.Module):
    """VQA model with symmetric (single shared block) cross-attention (baseline)."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3, freeze_encoders=True):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim, freeze=freeze_encoders)
        self.text_encoder = TextEncoder(embed_dim, freeze=freeze_encoders)
        self.fusion = SymmetricCrossModalFusion(embed_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(embed_dim, num_answers))

    def forward(self, images, input_ids, attention_mask):
        img = self.image_encoder(images)
        txt = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0

        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(img, txt, text_pad_mask)

        z = torch.cat([img_att.mean(dim=1), txt_att.mean(dim=1)], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}

---
## 3. Training

In [52]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, use_amp):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, input_ids, attention_mask, answers in tqdm(loader, desc="  train", leave=False):
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        answers = answers.to(device)

        optimizer.zero_grad()
        amp_ctx = autocast() if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(images, input_ids, attention_mask)
            loss = criterion(logits, answers)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * answers.size(0)
        correct += (logits.argmax(dim=1) == answers).sum().item()
        total += answers.size(0)

    return {"train_loss": total_loss / total, "train_acc": correct / total * 100}


@torch.no_grad()
def evaluate(model, loader, criterion, use_amp):
    model.eval()
    total_loss = 0.0
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    for images, input_ids, attention_mask, answers in tqdm(loader, desc="  eval", leave=False):
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        answers = answers.to(device)

        amp_ctx = autocast() if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(images, input_ids, attention_mask)
            loss = criterion(logits, answers)

        total_loss += loss.item() * answers.size(0)
        correct_top1 += (logits.argmax(dim=1) == answers).sum().item()
        _, top5 = logits.topk(5, dim=1)
        correct_top5 += (top5 == answers.unsqueeze(1)).any(dim=1).sum().item()
        total += answers.size(0)

    return {
        "val_loss": total_loss / total,
        "val_top1": correct_top1 / total * 100,
        "val_top5": correct_top5 / total * 100,
    }

In [53]:
def run_training(model_type, run_name=None):
    """Train a VQA model and return (model, history)."""
    set_seed(SEED)
    if run_name is None:
        run_name = f"{model_type}_s{SEED}"

    # Build model
    if model_type == "asymmetric":
        model = AsymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, FREEZE_ENCODERS)
    else:
        model = SymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, FREEZE_ENCODERS)
    model = model.to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel: {model_type} | Trainable params: {trainable:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    use_amp = USE_AMP and device.type == "cuda"
    scaler = GradScaler() if use_amp else None

    history = []
    best_val_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, use_amp)
        val_m = evaluate(model, val_loader, criterion, use_amp)
        elapsed = time.time() - t0

        epoch_data = {"epoch": epoch, **train_m, **val_m, "elapsed_s": round(elapsed, 1)}
        history.append(epoch_data)

        print(f"  Epoch {epoch}/{EPOCHS} | loss {train_m['train_loss']:.4f} | "
              f"train {train_m['train_acc']:.2f}% | val {val_m['val_top1']:.2f}% | "
              f"top5 {val_m['val_top5']:.2f}% | {elapsed:.0f}s")

        # Save checkpoint every epoch
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics": epoch_data,
        }, CHECKPOINT_DIR / f"{run_name}_epoch{epoch}.pt")

        if val_m["val_top1"] > best_val_acc:
            best_val_acc = val_m["val_top1"]
            torch.save({"model_state_dict": model.state_dict(), **epoch_data},
                       CHECKPOINT_DIR / f"{run_name}_best.pt")
            print(f"    New best: {best_val_acc:.2f}%")

    # Save history
    with open(METRICS_DIR / f"{run_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"Training complete. Best val_top1: {best_val_acc:.2f}%")
    return model, history

### 3.1 Train Symmetric Baseline

In [ ]:
symmetric_model, symmetric_history = run_training("symmetric")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2539.16it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model: symmetric | Trainable params: 4,978,664


  train:   0%|          | 3/6065 [00:28<15:17:14,  9.08s/it]

### 3.2 Train Asymmetric Model

In [ ]:
asymmetric_model, asymmetric_history = run_training("asymmetric")

---
## 4. Evaluation

In [ ]:
criterion = nn.CrossEntropyLoss()
use_amp = USE_AMP and device.type == "cuda"

sym_metrics = evaluate(symmetric_model, val_loader, criterion, use_amp)
asym_metrics = evaluate(asymmetric_model, val_loader, criterion, use_amp)

results = {"Symmetric": sym_metrics, "Asymmetric": asym_metrics}

# Print comparison table
metrics_keys = ["val_top1", "val_top5", "val_loss"]
header = "| Method       | " + " | ".join(k.replace('_', ' ').title() for k in metrics_keys) + " |"
sep    = "|--------------|" + "|".join("----------" for _ in metrics_keys) + "|"
print(header)
print(sep)
for name, vals in results.items():
    row = f"| {name:<12} | " + " | ".join(f"{vals[k]:>8.2f}" for k in metrics_keys) + " |"
    print(row)

---
## 5. Visualization

### 5.1 Training Curves

In [ ]:
histories = {"Symmetric": symmetric_history, "Asymmetric": asymmetric_history}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, hist in histories.items():
    epochs = [h["epoch"] for h in hist]
    axes[0].plot(epochs, [h["train_loss"] for h in hist], label=f"{name} train")
    axes[0].plot(epochs, [h["val_loss"] for h in hist], "--", label=f"{name} val")
    axes[1].plot(epochs, [h["train_acc"] for h in hist], label=f"{name} train")
    axes[1].plot(epochs, [h["val_top1"] for h in hist], "--", label=f"{name} val")
    axes[2].plot(epochs, [h["val_top5"] for h in hist], label=name)

axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss"); axes[0].legend()
axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="Top-1 Accuracy"); axes[1].legend()
axes[2].set(xlabel="Epoch", ylabel="Accuracy (%)", title="Top-5 Accuracy"); axes[2].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Comparison Bar Chart

In [ ]:
metric_names = ["val_top1", "val_top5"]
model_names = list(results.keys())
x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
for i, name in enumerate(model_names):
    values = [results[name][m] for m in metric_names]
    bars = ax.bar(x + i * width, values, width, label=name)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([m.replace("_", " ").title() for m in metric_names])
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model Comparison")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "comparison_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Attention Heatmaps

Visualize what the asymmetric model attends to: which image regions light up for each question word, and which words are most important for each image patch.

In [ ]:
# Visualization helpers
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])


def denormalize(img_tensor):
    """Convert normalised (3,H,W) tensor to (H,W,3) uint8 array."""
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * STD + MEAN
    return np.clip(img * 255, 0, 255).astype(np.uint8)


def decode_tokens(input_ids):
    """Decode token IDs to readable strings."""
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    return [tokenizer.decode(tid) for tid in input_ids.tolist()]


@torch.no_grad()
def get_attention_weights(model, image, input_ids, attention_mask):
    """Run a single sample and return attention weight dicts."""
    model.eval()
    img = image.unsqueeze(0).to(device)
    ids = input_ids.unsqueeze(0).to(device)
    mask = attention_mask.unsqueeze(0).to(device)
    _, attn = model(img, ids, mask)
    return {k: v.cpu() for k, v in attn.items()}


def plot_image_attention(image, attn_txt_to_img, question, tokens=None, top_tokens=4):
    """Overlay text->image attention heatmaps on the original image."""
    img_np = denormalize(image)
    attn = attn_txt_to_img.squeeze(0).numpy()  # (N_txt, N_img)

    grid_size = int(np.sqrt(attn.shape[1] - 1))  # 14 for ViT-B/16
    attn_spatial = attn[:, 1:]  # drop CLS column

    combined = attn_spatial.mean(axis=0).reshape(grid_size, grid_size)
    combined_resized = np.array(
        Image.fromarray(combined).resize((224, 224), Image.BILINEAR))

    token_importance = attn_spatial.sum(axis=1)
    top_idx = token_importance.argsort()[-top_tokens:][::-1]

    n_cols = min(top_tokens, len(top_idx)) + 1
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    if n_cols == 1:
        axes = [axes]

    axes[0].imshow(img_np)
    axes[0].imshow(combined_resized, alpha=0.5, cmap="jet")
    axes[0].set_title("Combined")
    axes[0].axis("off")

    for i, idx in enumerate(top_idx):
        if i + 1 >= len(axes):
            break
        token_attn = attn_spatial[idx].reshape(grid_size, grid_size)
        token_resized = np.array(
            Image.fromarray(token_attn).resize((224, 224), Image.BILINEAR))
        label = tokens[idx] if tokens else f"token {idx}"
        axes[i + 1].imshow(img_np)
        axes[i + 1].imshow(token_resized, alpha=0.5, cmap="jet")
        axes[i + 1].set_title(f'"{label}"')
        axes[i + 1].axis("off")

    fig.suptitle(question, fontsize=12)
    fig.tight_layout()
    return fig


def plot_text_attention(attn_img_to_txt, tokens, question):
    """Bar chart of image->text attention per token."""
    attn = attn_img_to_txt.squeeze(0).numpy()
    token_weights = attn.mean(axis=0)

    fig, ax = plt.subplots(figsize=(6, max(3, len(tokens) * 0.35)))
    y_pos = np.arange(len(tokens))
    ax.barh(y_pos, token_weights, color="steelblue")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Mean attention weight")
    ax.set_title(f"Image -> Text attention\n{question}")
    fig.tight_layout()
    return fig

In [ ]:
# Generate attention maps for sample validation images
for i in range(min(5, len(val_ds))):
    image, input_ids, attention_mask, answer_idx = val_ds[i]
    question = val_ds.samples[i]["question"]
    tokens = decode_tokens(input_ids)

    attn = get_attention_weights(asymmetric_model, image, input_ids, attention_mask)

    fig = plot_image_attention(image, attn["txt_to_img"], question, tokens)
    fig.savefig(FIGURES_DIR / f"attn_img_{i}.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig = plot_text_attention(attn["img_to_txt"], tokens, question)
    fig.savefig(FIGURES_DIR / f"attn_txt_{i}.png", dpi=150, bbox_inches="tight")
    plt.show()

### 5.4 Qualitative Comparison Grid

Side-by-side predictions and attention maps for both models on the same samples.

In [ ]:
@torch.no_grad()
def qualitative_grid(models, dataset, idx_to_answer, n_samples=6, save_path=None):
    """Grid of qualitative examples comparing models."""
    sample_indices = torch.randperm(len(dataset))[:n_samples].tolist()
    model_names = list(models.keys())
    n_cols = 1 + len(model_names)
    n_rows = len(sample_indices)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(sample_indices):
        image, input_ids, attention_mask, answer_idx = dataset[idx]
        question = dataset.samples[idx]["question"]
        gt_answer = idx_to_answer[answer_idx.item()]
        img_np = denormalize(image)

        # Column 0: original image + question
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f"Q: {question}\nGT: {gt_answer}", fontsize=9)
        axes[row, 0].axis("off")

        # Remaining columns: one per model
        for col, name in enumerate(model_names, start=1):
            model = models[name]
            attn = get_attention_weights(model, image, input_ids, attention_mask)

            img_t = image.unsqueeze(0).to(device)
            ids_t = input_ids.unsqueeze(0).to(device)
            mask_t = attention_mask.unsqueeze(0).to(device)
            logits, _ = model(img_t, ids_t, mask_t)
            pred_answer = idx_to_answer.get(logits.argmax(dim=1).item(), "???")

            # Attention heatmap
            attn_t2i = attn["txt_to_img"].squeeze(0).numpy()
            grid_size = int(np.sqrt(attn_t2i.shape[1] - 1))
            combined = attn_t2i[:, 1:].mean(axis=0).reshape(grid_size, grid_size)
            combined_resized = np.array(
                Image.fromarray(combined).resize((224, 224), Image.BILINEAR))

            axes[row, col].imshow(img_np)
            axes[row, col].imshow(combined_resized, alpha=0.5, cmap="jet")
            marker = "correct" if pred_answer == gt_answer else "wrong"
            axes[row, col].set_title(f"{name}\nPred: {pred_answer} ({marker})", fontsize=9)
            axes[row, col].axis("off")

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig

In [ ]:
models_dict = {"Asymmetric": asymmetric_model, "Symmetric": symmetric_model}

fig = qualitative_grid(
    models_dict, val_ds, idx_to_answer, n_samples=6,
    save_path=str(FIGURES_DIR / "qualitative_grid.png"))
plt.show()